In [1]:
import os
import PIL
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import time


In [2]:
batch_size = 64
learning_rate = 1e-3
epochs = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(root='asl_alphabet_train\\asl_alphabet_train', transform=transform)
test_dataset = datasets.ImageFolder(root='asl_alphabet_train\\asl_alphabet_train', transform=transform)

train_size = int(len(train_dataset))
test_size =  len(test_dataset)

# train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Total samples: {len(train_dataset)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

train_images, train_labels = next(iter(train_loader))
test_images, test_labels = next(iter(test_loader))

print("Training batch shape:", train_images.shape)
print("Testing batch shape:", test_images.shape)

class_names = train_dataset.classes 
print("Classes:", class_names)
num_classes = len(class_names)

Total samples: 87000
Training samples: 87000
Testing samples: 87000
Training batch shape: torch.Size([64, 3, 200, 200])
Testing batch shape: torch.Size([64, 3, 200, 200])
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [3]:
class_to_label = {class_name: idx for idx, class_name in enumerate(train_dataset.classes)}
label_to_class = {idx: class_name for class_name, idx in class_to_label.items()}

print("Class-to-Label Mapping:", class_to_label)
print("Label-to-Class Mapping:", label_to_class)

Class-to-Label Mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 'del': 26, 'nothing': 27, 'space': 28}
Label-to-Class Mapping: {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'J', 10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'O', 15: 'P', 16: 'Q', 17: 'R', 18: 'S', 19: 'T', 20: 'U', 21: 'V', 22: 'W', 23: 'X', 24: 'Y', 25: 'Z', 26: 'del', 27: 'nothing', 28: 'space'}


In [4]:
class ImageClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5), #output_shape=(batch_size, 16, 196, 196)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), #output_shape=(batch_size, 16, 98, 98)
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5), #output_shape=(batch_size, 32, 94, 94)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), #output_shape=(batch_size, 32, 47, 47)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5), #output_shape=(batch_size, 64, 43, 43)
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=5), #output_shape=(batch_size, 128, 39, 39)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), #output_shape=(batch_size, 128, 19, 19)
            nn.Dropout(p=0.2),
            nn.Flatten(),
            nn.Linear(128 * 19 * 19, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )
        
    def forward(self, x):
        logits = self.convolution(x)
        return logits
    
model = ImageClassifier(num_classes=num_classes).to(device)

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer):
    train_losses_list = []
    train_total_correct = 0
    train_total_samples = 0

    model.train()

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device, dtype=torch.long)

        pred = model(x)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_losses_list.append(loss.item())

        train_total_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_train_loss = sum(train_losses_list) / len(train_losses_list)
    avg_train_acc = train_total_correct / len(dataloader.dataset)

    return avg_train_loss, avg_train_acc

In [7]:
def test_loop(dataloader, model, loss_fn):
    
    test_losses_list = []
    test_total_correct = 0
    test_total_samples = 0
    model.eval()
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device=device, dtype=torch.long)
            
            pred = model(x)
            loss = loss_fn(pred, y)
            
            test_losses_list.append(loss.item())

            test_total_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_test_loss = sum(test_losses_list) / len(test_losses_list)
    avg_test_acc = test_total_correct / len(dataloader.dataset)
    return avg_test_loss, avg_test_acc

In [8]:
for epoch in range(epochs):
    start = time.time()
    avg_train_loss, avg_train_acc = train_loop(train_loader, model, loss_fn, optimizer)
    avg_test_loss, avg_test_acc = test_loop(test_loader, model, loss_fn)
    
    print(f"Epoch: {epoch+1}/{epochs} - {round(time.time()-start)}s - Training accuracy: {avg_train_acc:.4f} - Training loss: {avg_train_loss:.4f} - Test accuracy: {avg_test_acc:.2f} - Test loss: {avg_test_loss:.2f}")

KeyboardInterrupt: 